# 01 — Exploring the California Housing Dataset

**Goal of this notebook:** understand the data *before* any machine learning happens.

By the end you should be able to explain:

- what one row of the dataset represents;
- what every column means (and its units);
- which column is the **target** (`y`) and which are **features** (`X`);
- whether anything is missing or suspicious.

> ⚠️ **We do not train any model in this notebook.** Understanding the data is the first milestone.

**How to use this notebook**

1. Read each explanation.
2. Where it says **🔮 Predict first**, write down your guess *before* running the cell.
3. Run the cell (`Shift + Enter`) and compare.

Companion reading: [`docs/00-ml-big-picture.md`](../docs/00-ml-big-picture.md) (before), [`docs/01-understanding-the-data.md`](../docs/01-understanding-the-data.md) and [`docs/02-features-and-target.md`](../docs/02-features-and-target.md) (after).

## 1. Imports

- **pandas** (`pd`) works with tables of data, called *DataFrames*.
- **`fetch_california_housing`** is scikit-learn's built-in loader for our dataset.
- `pd.set_option(...)` just makes wide tables print nicely.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 2. Load the dataset

The first time this runs, scikit-learn downloads the data (about 400 KB) and caches it in `~/scikit_learn_data`. After that it loads from the cache.

`as_frame=True` asks for pandas tables (with column names) instead of plain NumPy arrays.

In [ ]:
housing = fetch_california_housing(as_frame=True)

### What *is* `housing`?

`housing` is a container holding several things. Let's check the type of the container and of two things inside it.

🔮 **Predict first:** `housing.data` holds the 8 feature columns, and `housing.target` holds only the target column. Which one do you think is a table (DataFrame) and which is a single column (Series)?

In [ ]:
print(type(housing))
print(type(housing.data))
print(type(housing.target))

- **`Bunch`**: a scikit-learn container that works like a dictionary (`housing.data` is the same as `housing["data"]`).
- **`DataFrame`**: a table with rows and named columns. `housing.data` holds the **8 feature columns**.
- **`Series`**: a single column. `housing.target` holds the **target**, `MedHouseVal`.

The dataset also comes with a written description. Reading the documentation of a dataset is a habit worth building:

In [ ]:
print(housing.DESCR[:1800])

## 3. One table with everything: `df`

`housing.frame` is a single DataFrame with the 8 features **and** the target side by side. We'll call it `df` (short for DataFrame, a common convention).

This project also has its own small loader, `house_price.data.load_housing_data()`, that returns exactly the same table. The second line below checks that the two are identical.

In [ ]:
from house_price.data import load_housing_data

df = housing.frame
print(df.equals(load_housing_data()))

## 4. What does one row represent?

This is the most important fact about this dataset:

> **Each row is one California *census block group*, not one individual house.**

A census block group is a small neighbourhood used by the US Census Bureau, typically home to 600–3,000 people. The data comes from the **1990 US Census**.

That's why many columns are **averages** or **medians** for the whole neighbourhood. When we later predict `MedHouseVal`, we predict the **median house value of a neighbourhood**, not the price of one specific house.

### What does each column mean?

| Column | Meaning | Unit |
|---|---|---|
| `MedInc` | Median household income in the block group | tens of thousands of \$ (3.5 ≈ \$35,000) |
| `HouseAge` | Median age of the houses | years |
| `AveRooms` | Average number of rooms per household | rooms |
| `AveBedrms` | Average number of bedrooms per household | bedrooms |
| `Population` | Number of people living in the block group | people |
| `AveOccup` | Average number of people per household | people |
| `Latitude` | How far north (higher = further north) | degrees |
| `Longitude` | How far west (more negative = further west) | degrees |
| **`MedHouseVal`** | **Median house value: the target** | **hundreds of thousands of \$** (2.5 ≈ \$250,000) |

## 5. `df.head()`: what does the data look like?

`head()` shows the **first 5 rows**. It's the quickest way to get a feel for a table.

After running it, pick one row and try to say it as a sentence: *"In this neighbourhood, the median income is about \$…, the houses are about … years old, …"*

In [ ]:
df.head()

**What to notice:**

- the numbers on the far left (0, 1, 2, …) are the **index**, the row labels;
- all values are numbers;
- row 0 has `MedHouseVal = 4.526`, which means about **\$452,600** (4.526 × \$100,000);
- the first rows all have similar latitude/longitude: they're all in the San Francisco Bay Area. `df.sample(5)` shows random rows instead.

## 6. `df.shape`: how big is the data?

`shape` gives **(number of rows, number of columns)**. Note that there are no parentheses after `shape`: it's an attribute, not a method.

🔮 **Predict first:** how many rows and columns do you expect?

In [ ]:
df.shape

**20,640 rows** (block groups) × **9 columns** (8 features + 1 target).

## 7. `df.columns`: what information do we have?

In [ ]:
df.columns

## 8. `df.info()`: types and missing values

`info()` prints one line per column with:

- **Non-Null Count**: how many values are *not* missing;
- **Dtype**: the data type (`float64` = decimal number).

🔮 **Predict first:** do you expect any column to have fewer than 20,640 non-null values?

In [ ]:
df.info()

**What to notice:** every column has **20,640 non-null** values (no missing data) and type **`float64`** (all numeric). If a column you expected to be numeric showed `object`, that would usually mean some values are text, which is a sign of dirty data.

## 9. `df.describe()`: typical values and extremes

`describe()` summarises every numeric column:

| Row | Meaning |
|---|---|
| `count` | number of non-missing values |
| `mean` | the average |
| `std` | standard deviation: roughly how far a typical value is from the average |
| `min` / `max` | smallest / largest value |
| `25%` / `50%` / `75%` | a quarter / half / three quarters of values are below this (`50%` is the **median**) |

`.round(2)` just makes the output easier to read.

In [ ]:
df.describe().round(2)

**Questions to ask yourself while reading this table:**

1. What's the minimum and maximum `MedHouseVal`, in dollars?
2. Is the mean of `MedHouseVal` higher or lower than its median (`50%`)? What might that mean?
3. Look at the `max` of `AveRooms`, `AveBedrms` and `AveOccup`. Do those values make sense for a normal household?
4. Look at the `max` of `HouseAge` and `MedHouseVal`. Are they suspiciously round?

## 10. `df.isna().sum()`: missing values, column by column

- `df.isna()` builds a table of `True`/`False`: `True` where a value is missing.
- `.sum()` counts the `True` values in each column (Python treats `True` as 1).

In [ ]:
df.isna().sum()

All zeros: **no missing values**. That's unusual for real-world data. This dataset has been cleaned for teaching.

But "no missing values" doesn't mean "nothing strange". Let's look more closely.

## 11. Suspicious values

Two things stood out in `describe()`:

1. **Very large averages.** Some block groups have huge `AveRooms`, `AveBedrms` or `AveOccup` values. The dataset description explains that block groups with few households (e.g. holiday resorts, dormitories) can produce surprisingly large averages.
2. **Capped values.** `MedHouseVal` stops at about 5.0 (\$500,000) and `HouseAge` at 52. Values above those limits were recorded *as* the limit, so "5.0" really means "\$500,000 **or more**".

In [ ]:
# How many block groups sit exactly at the maximum value of each column?
for column in ["MedHouseVal", "HouseAge"]:
    at_max = (df[column] == df[column].max()).sum()
    print(f"{column}: max = {df[column].max()}, rows at the max = {at_max}")

In [ ]:
# The single most extreme household-occupancy value.
df.loc[df["AveOccup"].idxmax()]

We **won't change** any of these values now. We're just noting them. Later, they may explain some of the model's biggest mistakes.

## 12. The target and its units

`MedHouseVal` is the value we want to predict: the **target**. It's measured in units of **\$100,000**:

```text
MedHouseVal = 2.5   →   2.5 × $100,000 = $250,000
```

Always convert to dollars when you talk about results.

In [ ]:
(df["MedHouseVal"].head() * 100_000).round()

## 13. Introducing `X` and `y`

In machine learning we separate:

- **`X`**, the **features**: the information we give the model;
- **`y`**, the **target**: the answer we want the model to learn to predict.

scikit-learn has already separated them for us:

In [ ]:
X = housing.data
y = housing.target

print("X shape:", X.shape)
print("y shape:", y.shape)

```text
X  →  20,640 rows × 8 features   (information about each block group)
y  →  20,640 answers              (one MedHouseVal per block group)
```

`X` and `y` have the **same number of rows**: every block group has exactly one answer.

Why must `MedHouseVal` **not** be inside `X`? Because when we later ask the model to predict the value of a new area, we won't *have* that value. If we did, we wouldn't need a model. (See [`docs/02-features-and-target.md`](../docs/02-features-and-target.md).)

> 🛑 **We stop here.** We have `X` and `y`, but we do **not** train anything yet. First make sure you understand the data.

## 14. Exercises

Answer these in the empty cells below (code) or in a Markdown cell (words). They're the **First Concrete Exercise** from the project brief.

1. What type is `housing`?
2. What type is `housing.data`?
3. What type is `housing.target`?
4. What is `df.shape`?
5. What are the column names?
6. What does one row represent?
7. Which column is `y`?
8. Which columns could be `X`?
9. What unit is `MedHouseVal` measured in?
10. Are there missing values?

**Extra exercises:**

- **A.** Show 5 *random* rows with `df.sample(5, random_state=0)`. Describe one of them in a plain-English sentence, converting income and house value to dollars.
- **B.** Build your own `X` containing only `MedInc`, `HouseAge` and `AveRooms`, and a `y` containing `MedHouseVal`, starting from `df`. Predict the shapes before printing them. *(Hint: a **list** of column names inside the brackets gives a DataFrame.)*
- **C.** What percentage of block groups have `MedHouseVal` at its capped maximum?
- **D.** Write down the three features you think will be most useful for predicting `MedHouseVal`, and why. You'll test these hypotheses later.

In [ ]:
# Exercise A — your code here


In [ ]:
# Exercise B — your code here


In [ ]:
# Exercise C — your code here


## ✅ Checkpoint

Before continuing, you should be able to explain **without looking**:

- □ What does one row of this dataset represent?
- □ What does each column mean, and what are the units of `MedInc` and `MedHouseVal`?
- □ What do `head()`, `shape`, `info()`, `describe()` and `isna().sum()` each tell you?
- □ Which column is the target, and why can't it be one of the features?
- □ Why do `X` and `y` have the same number of rows?

**Next:** read [`docs/01-understanding-the-data.md`](../docs/01-understanding-the-data.md) and [`docs/02-features-and-target.md`](../docs/02-features-and-target.md). Stop there until these concepts feel solid: model training comes later.